In [1]:

from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(
    model="gapgpt-qwen-3.5",
    temperature=0.3,
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_KEY")
)

INPUT_FILE = "../inputs/HD5L.txt"

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from libAgent.markdownSplitter import markdownTextSplitter
chunks = markdownTextSplitter(INPUT_FILE)

print(len(chunks))

231


In [11]:
print( llm.invoke('hello') )

content='Hello! How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3.6-35B-A3B-FP8', 'system_fingerprint': None, 'id': 'chatcmpl-ce0e70248fc21c84b4e78213ad114c50', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f02b0-95b0-76a3-a4d2-e5df1bc5af50-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {}, 'output_token_details': {}}


#Extractor

In [10]:
print(chunks[6])

page_content='| Revised Chapter   | Revised Contents                                                                          |
|-------------------|-------------------------------------------------------------------------------------------|
| Chapter 6         | • Reserved the 13th function of F12.20 • Add Group F19, F20.15 - F20.19 • Reserved F17.02 |' metadata={'h2': 'Version: V2.5'}


In [19]:
from pydantic import BaseModel, Field
from typing import List

class Extracted(BaseModel):
    main_topics: List[str] = Field(
        default_factory=list,
        description="High-level topics explicitly present in the text"
    )

    subtopics: List[str] = Field(
        default_factory=list,
        description="Detailed concepts under main topics"
    )

    entities: List[str] = Field(
        default_factory=list,
        description="Named entities like devices, error codes, APIs"
    )

    keywords: List[str] = Field(
        default_factory=list,
        description="Searchable terms for retrieval"
    )

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
        (
        "system",
        """
        You are a knowledge extraction system.

        Analyze the provided document chunk and extract:

        1. Main Topics
        - High-level subjects discussed in the chunk.
        - Use concise canonical names.
        - Maximum 5 items.

        2. Subtopics
        - More specific concepts under the main topics.
        - Maximum 10 items.

        3. Entities
        - Products
        - Devices
        - Components
        - Technologies
        - Standards
        - Organizations
        - Error codes
        - Commands
        - Protocols
        - Any named technical concepts

        4. Keywords
        - Important searchable terms.
        - Include technical terminology.
        - Include abbreviations when relevant.
        - Maximum 20 items.

        Rules:
        - Do not invent information.
        - Extract only information explicitly present in the text.
        - Remove duplicates.
        - Use canonical names whenever possible.
        - Return JSON only.

        Output schema:
        
        "main_topics": [],
        "subtopics": [],
        "entities": [],
        "keywords": []

        """
    ),
    (
        "user",
        """
        Document Chunk:

        {chunk}
        """
    ),
])

llmExtranct = llm.with_structured_output(Extracted)
chain  = prompt | llmExtranct


import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(text))

MIN_TOKENS = 500

merged_chunks = []

i = 0
while i < len(chunks):
    current = chunks[i].page_content
    tokens = count_tokens( current )

    while tokens < MIN_TOKENS and i + 1 < len(chunks):
        i += 1
        current += "\n\n" + chunks[i].page_content
        tokens = count_tokens(current)

    merged_chunks.append(current)
    i += 1

print(len(merged_chunks))

extraxted = []
for i in range(0,len(merged_chunks)):
    extraxted.append( chain.invoke({'chunk':merged_chunks[i]}) )


72


In [35]:
extraxtedJson = [ item.json() for item in extraxted ]
import json
with open('./extraxted.json','w' , encoding="utf-8") as f:
    f.write(json.dumps(extraxtedJson , ensure_ascii=False  , indent=2))

In [36]:
print(extraxted[0].json())

{"main_topics":["Product Overview","Installation Guidelines","Operation Instructions","Parameter Configuration","Application Guidance"],"subtopics":["Safety Precautions","Model Identification","Mechanical Installation","Electrical Wiring","EMC Compliance","Keypad Operation","Function Parameters","Encoder Interfaces","Elevator Commissioning","Troubleshooting"],"entities":["HD5L Series","HP","marketing@hpmont.com","Technical Service Center","Keypad","Control Board","I/O Board","Encoder Interface Board","HD-PG2-OC-FD","HD-PG5-SINCOS-FD","HD-PG6-UVW-FD","HD-PG11-SC-FD","HD5L-PG1-SC","EMI Filter","Reactor","Braking Resistor","Power Regenerative Unit","F12.20","F19","F20.15 - F20.19","F17.02","Group D","Group F","Group Y","D00","D01","D02","D03","D04","F00","F01","F02","F03","F04","F05","F06","F07","F08","F09","F10","F11","F12","F13","F14","F15","F16","F17","F18","F19","F20","PG Parameters","SCI Communication","EMC","SINCOS","UVW","SC"],"keywords":["HD5L Series","User Manual","Technical Serv

In [37]:

promptMerge = ChatPromptTemplate.from_messages([
        (
        "system",
        """
        You are a knowledge aggregation system.

        You will receive multiple JSON outputs extracted from document chunks.

        Your task is to merge them into a unified knowledge representation.

        ### You must produce:

        1. Main Topics
        - Consolidate and deduplicate all main topics
        - Keep only the most important high-level topics
        - Max 10 items

        2. Subtopics
        - Merge all subtopics
        - Remove duplicates and overlaps
        - Keep only meaningful specific concepts
        - Max 30 items

        3. Entities
        - Merge all entities
        - Normalize names (same entity must appear once)
        - Keep important technical entities only

        4. Keywords
        - Merge and deduplicate keywords
        - Keep only high-signal searchable terms
        - Max 50 items

        ### Important rules:
        - Do NOT invent new information
        - Do NOT lose rare but important technical entities (e.g., error codes, protocol names)
        - Prefer canonical names
        - If two items are similar, merge them into one general term
        - Output must be valid JSON only

        ### Output schema:

        "main_topics": [],
        "subtopics": [],
        "entities": [],
        "keywords": []

        """
    ),

    (
        "user",
        """
        ### Input data:

        {chunks}
        """
    ),
])

chain  = promptMerge | llmExtranct

extraxtedMerge = []

MERGE_SIZE = 5
for i in range(0, len(extraxtedJson), MERGE_SIZE ):
    extraxtedMerge.append( chain.invoke({ 'chunks':extraxtedJson[i:i+MERGE_SIZE] } ).json() )



In [39]:
print( len(extraxtedMerge))

15


In [43]:
promptProfile = ChatPromptTemplate.from_messages([
        (
        "system",
        """
You are an expert knowledge base profiling system.

You will receive aggregated metadata extracted from an entire document collection.

Your task is to generate a concise, high-quality description of this knowledge base for an AI agent that must decide whether to use this RAG tool.

The description should help the agent understand:

- What domain this knowledge base covers.
- What kinds of questions it is capable of answering.
- What information users can expect to retrieve.
- The scope and limitations of the knowledge base.
- The terminology and concepts that are well represented.

Do NOT enumerate every topic.
Instead, synthesize them into a natural, coherent description.

Requirements:

- Write one concise paragraph (100–200 words).
- Write in neutral, factual language.
- Mention the major domains naturally.
- Mention typical user intents (e.g. troubleshooting, configuration, API usage, installation, architecture, specifications, error diagnosis, best practices, compatibility, etc.) only if they are supported by the input.
- Do not invent information.
- Do not repeat similar concepts.
- Do not mention that the information came from metadata.
- The description should read like the official description of a search tool.


        """
    ),
    (
        "user",
        """
        ### Input data:
        {merged_metadata}
        """
    )
])

from langchain_core.output_parsers import StrOutputParser

chainProfile = promptProfile | llm | StrOutputParser()

print( chainProfile.invoke({'merged_metadata':extraxtedMerge }) )

This knowledge base provides comprehensive technical documentation for the Hpmont HD5L Series elevator drive controller. It covers the full lifecycle of the device, including mechanical installation, electrical wiring, and safety compliance with IEC standards. The content details hardware specifications for I/O boards, encoder interfaces (SINCOS, UVW, ABZ), and peripheral components like braking resistors and reactors. A significant portion is dedicated to advanced parameter configuration, focusing on motor control algorithms such as V/f, Sensorless Vector Control (SVC), and Closed-loop Vector Control, with specific tuning for asynchronous and synchronous motors. It also extensively documents the MODBUS communication protocol, including RTU/ASCII frame structures and register mapping. Users can retrieve information on troubleshooting fault codes, managing elevator-specific features like inspection and battery-driven modes, performing auto-tuning, and adhering to maintenance and EMC gui